# Chapter 2.2: LangChain Runnables & LCEL

In LangChain, the **Runnable protocol** (LangChain Expression Language - LCEL) provides a unified interface for composing custom chains.

### Standard Methods supported by all Runnables:
- `invoke`: Call the runnable on a single input
- `stream`: Stream back chunks of the response
- `batch`: Call the runnable on a list of inputs
- `ainvoke`, `astream`, `abatch`: Asynchronous versions

### 1. Environment Setup & Model Initialization

In [ ]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough

load_dotenv()

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash"
)

### 2. Basic LCEL Chain (Prompt | LLM | StrOutputParser)

In [ ]:
prompt = ChatPromptTemplate.from_template("Explain {topic} in two simple sentences for a beginner.")
parser = StrOutputParser()

chain = prompt | llm | parser

response = chain.invoke({"topic": "LangChain Runnables"})
print(response)

### 3. RunnableLambda: Wrapping custom functions

In [ ]:
def word_count(text: str) -> dict:
    words = len(text.split())
    return {"text": text, "word_count": words}

count_runnable = RunnableLambda(word_count)

extended_chain = chain | count_runnable
result = extended_chain.invoke({"topic": "Quantum Computing"})
print(result)

### 4. RunnableParallel: Running branches in parallel

In [ ]:
joke_prompt = ChatPromptTemplate.from_template("Tell a short one-liner joke about {topic}.")
poem_prompt = ChatPromptTemplate.from_template("Write a 2-line rhyming poem about {topic}.")

joke_chain = joke_prompt | llm | parser
poem_chain = poem_prompt | llm | parser

parallel_chain = RunnableParallel(
    joke=joke_chain,
    poem=poem_chain
)

res = parallel_chain.invoke({"topic": "Software Engineering"})
print("--- JOKE ---")
print(res["joke"])
print("\n--- POEM ---")
print(res["poem"])